# Разбор A. Модель «структура — свойство»

Дескрипторы можно объединить в модель свойства, но небольшое число лекарственных объектов не образует обучающую выборку. Вернёмся к ESOL как отдельному опубликованному контролю QSPR. Гипотеза: линейная модель дескрипторов превосходит среднее обучающей выборки на отложенных молекулах. Вывод о растворимости A/B потребует проверки области применимости.

Из всех ESOL SMILES вычислите MolWt, MolLogP, TPSA, HBD, HBA. Группируйте по каноническому isomeric SMILES, используйте GroupShuffleSplit с seed=42 и test_size=.2. Сравните Ridge pipeline с DummyRegressor. Сохраните ID train/test и проверьте отсутствие пересечения групп. Сделайте диаграмму y–prediction.

In [ ]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M12-L01','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors,Lipinski
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
rows=esol();mols=[Chem.MolFromSmiles(r['smiles']) for r in rows];assert all(m is not None for m in mols)
def features(m):return [Descriptors.MolWt(m),Descriptors.MolLogP(m),Descriptors.TPSA(m),Lipinski.NumHDonors(m),Lipinski.NumHAcceptors(m)]
X=np.array([features(m) for m in mols]);y=np.array([r['measured'] for r in rows])
groups=np.array([Chem.MolToSmiles(m,isomericSmiles=True) for m in mols])
train,test=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(X,y,groups))
assert not set(groups[train])&set(groups[test])


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [ ]:
model=make_pipeline(StandardScaler(),Ridge(alpha=1));model.fit(X[train],y[train]);pred=model.predict(X[test])
baseline=DummyRegressor().fit(X[train],y[train]).predict(X[test])
rmse=lambda a,b:float(np.sqrt(np.mean((a-b)**2)))
print({'Ridge_RMSE':rmse(y[test],pred),'baseline_RMSE':rmse(y[test],baseline)})
fig,ax=plt.subplots();ax.scatter(y[test],pred,s=10);ax.plot([-10,3],[-10,3],color='gray')
ax.set(xlabel='Measured logS',ylabel='Held-out predicted logS');publish_plot(fig,'qspr.png')


Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [ ]:
result={'kind':'calculated','seed':42,'split':'identity groups','train_ids':train.tolist(),'test_ids':test.tolist(),
        'test_RMSE_logS':rmse(y[test],pred),'baseline_RMSE_logS':rmse(y[test],baseline),'B_prediction':'NOT_VALIDATED'}


In [ ]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M12-L01','A',result,ROOT,inputs=['practicuum/shared/data/esol.csv'])
print('Versioned evidence:',receipt)


Даже улучшение над baseline не доказывает причинность дескрипторов или универсальный перенос. Не включайте опубликованный ESOL predicted в признаки: это чужая модель с неизвестной относительно вашего split историей. Далее проверим удалённость A и B от train.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?